# Stage 2 (Title Block) — Qwen zero-shot scoring, GPU-only

Minimal notebook: downloads the prepped, human/Claude-verified package (images + corrected
ground truth CSV) from HF, loads Qwen3-VL-8B base, scores it, pushes results, disconnects.
All CPU-side prep (data fetch, ground-truth drafting, correction) already happened locally on
the Mac — see `titleblock_corrected_v1.csv` / `titleblock_images_v1.zip` in
`timthy45/pnid-extraction-datasets`. GPT-5.5-low was already scored locally too (no GPU needed
for an API call) — see `gpt55low_titleblock_results.json`. This notebook exists ONLY to run
the one step that genuinely needs a GPU: Qwen inference.

## 1. Config

In [6]:
HF_TOKEN = "PASTE_HF_TOKEN_HERE"
DATA_REPO = "timthy45/pnid-extraction-datasets"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here"


## 2. Install

In [7]:
!pip install -q -U transformers accelerate huggingface_hub

import torch
print("CUDA available:", torch.cuda.is_available())


CUDA available: True


## 3. Download the prepped package (images + corrected ground truth)

In [8]:
import zipfile, time, json, csv, random, re
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)

def fetch_with_retry(filename, max_attempts=20, base_backoff_s=10, max_backoff_s=90):
    last_err = None
    for attempt in range(max_attempts):
        try:
            return hf_hub_download(repo_id=DATA_REPO, filename=filename,
                                   repo_type="dataset", token=HF_TOKEN)
        except Exception as e:
            last_err = e
            wait = min(max_backoff_s, base_backoff_s * (1.5 ** attempt)) + random.uniform(0, 3)
            print(f"  [retry {attempt+1}/{max_attempts}] {type(e).__name__}: "
                 f"{str(e)[:120]} - waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"download of {filename} failed after {max_attempts} retries") from last_err

csv_path = fetch_with_retry("benchmarks/titleblock_corrected_v1.csv")
zip_path = fetch_with_retry("benchmarks/titleblock_images_v1.zip")

IMG_DIR = DATA / "titleblock_images"
IMG_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(IMG_DIR)

TB_FIELDS = ["drawing_number", "revision", "title", "site"]

# Qwen-specific prompt, tuned to the exact failure modes observed in the first zero-shot run:
#   - revision: Qwen grabbed a sheet number ("sh03"), a phase label ("DRAFT 30% DESIGN
#     DRAWINGS"), and "1" instead of the REV table letter -> hard negative rules added.
#   - title/site: Qwen paraphrased/truncated instead of transcribing -> verbatim rule added.
TB_PROMPT = (
    "Look at the title block of this P&ID drawing sheet (the boxed region along the "
    "right edge or bottom-right corner).\n"
    "\n"
    "Extract these 4 fields. COPY ALL TEXT EXACTLY AS PRINTED, character for character. "
    "Do not shorten, paraphrase, or reword anything.\n"
    "\n"
    "1. drawing_number - the value in the box labeled DRAWING NO / DWG NO / SHEET NO "
    "identifier for this drawing (e.g. 51Y613, M-01, MG129-00010).\n"
    "2. revision - the revision letter or number of this drawing, from the box or column "
    "labeled REV / REV NO / REVISION. It is almost always a single letter (A, B, C) or "
    "single digit (0, 1, 2).\n"
    "   - It is NOT the sheet number (like sh03 or 3/7).\n"
    "   - It is NOT a phase or status note (like DRAFT 30% DESIGN DRAWINGS or PRELIMINARY).\n"
    "   - It is NOT the scale, date, or project number.\n"
    "   - If there is a revision history table, take the revision value of the newest row.\n"
    "   - If no REV box or table exists, use null.\n"
    "3. title - the drawing title exactly as printed, including every line of it (titles "
    "often span 2-3 stacked lines - include them all, joined with spaces). Include prefixes "
    "like INSTRUMENTATION and suffixes like P&ID if printed.\n"
    "4. site - the project name and site/address exactly as printed, including every line "
    "(e.g. project name line + street address line + city line, joined with commas). If "
    "only a company logo is present but no project/site text, use null.\n"
    "\n"
    "Respond with ONLY this JSON object and nothing else:\n"
    "{\"drawing_number\": ..., \"revision\": ..., \"title\": ..., \"site\": ...}\n"
    "Use null for any field genuinely not present. If this image is a symbol legend or "
    "reference diagram rather than a titled engineering drawing, use null for every field."
)
corrected_tb = {}
with open(csv_path) as f:
    for row in csv.DictReader(f):
        corrected_tb[row["sheet_id"]] = {k: (row[k] if row[k] != "" else None) for k in TB_FIELDS}

print(f"loaded {len(corrected_tb)} corrected sheets: {list(corrected_tb.keys())}")


loaded 6 corrected sheets: ['test_151', 'test_194', 'test_196', 'test_216', 'train_198', 'train_175']


## 4. Load Qwen3-VL-8B base (zero-shot, no adapter)

In [9]:
from transformers import AutoModelForImageTextToText, AutoProcessor
import random as _random_retry

def load_with_retry(loader_fn, max_attempts=20, base_backoff_s=10, max_backoff_s=90):
    last_err = None
    for attempt in range(max_attempts):
        try:
            return loader_fn()
        except Exception as e:
            last_err = e
            backoff = min(max_backoff_s, base_backoff_s * (1.5 ** attempt))
            wait = backoff + _random_retry.uniform(0, backoff * 0.3)
            print(f"  [retry {attempt+1}/{max_attempts}] {type(e).__name__}: "
                 f"{str(e)[:160]} - waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"failed after {max_attempts} retries - HF Xet-bridge "
                       "signing issue persisted longer than usual; rerun this cell") from last_err

processor = load_with_retry(lambda: AutoProcessor.from_pretrained(QWEN_MODEL_ID))
model = load_with_retry(lambda: AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")).eval()
print("Qwen base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def qwen_generate(image, prompt, max_tokens=300):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    t = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(t, skip_special_tokens=True).strip()


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Qwen base loaded. VRAM: 17.6 GB


## 5. Score (the only GPU-dependent step)

In [10]:
from PIL import Image

def parse_tb_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return {f: None for f in TB_FIELDS}
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return {f: None for f in TB_FIELDS}
    return {f: d.get(f) for f in TB_FIELDS}

def norm(v):
    if v is None:
        return None
    return str(v).strip().lower().replace(" ", "")

def tokens(v):
    if v is None:
        return set()
    return set(re.findall(r"[a-z0-9]+", str(v).lower()))

def overlap_ratio(pred, gt):
    pt, gt_t = tokens(pred), tokens(gt)
    if not gt_t:
        return 1.0 if not pt else 0.0
    return len(pt & gt_t) / len(gt_t)

# Two metrics, reported side by side:
#   exact - normalized string equality (strict; punishes benign paraphrase)
#   loose - exact OR >=60% of ground-truth tokens present in the prediction
#           (same metric used for the GPT-5.5-low numbers scored locally)
exact_correct = {f: 0 for f in TB_FIELDS}
loose_correct = {f: 0 for f in TB_FIELDS}
all_exact = 0
all_loose = 0
per_sheet = {}
for sheet_id, gt in corrected_tb.items():
    img = Image.open(IMG_DIR / f"{sheet_id}.png")
    raw = qwen_generate(img, TB_PROMPT, max_tokens=300)
    pred = parse_tb_json(raw)
    sheet_exact_ok, sheet_loose_ok = True, True
    field_status = {}
    for f in TB_FIELDS:
        e = norm(pred.get(f)) == norm(gt.get(f))
        l = e or overlap_ratio(pred.get(f), gt.get(f)) >= 0.6
        field_status[f] = {"pred": pred.get(f), "gt": gt.get(f), "exact": e, "loose": l}
        exact_correct[f] += e
        loose_correct[f] += l
        if not e: sheet_exact_ok = False
        if not l: sheet_loose_ok = False
    all_exact += sheet_exact_ok
    all_loose += sheet_loose_ok
    per_sheet[sheet_id] = field_status
    print(f"  {sheet_id}: {'ALL OK' if sheet_exact_ok else 'mismatch'} - {field_status}")

n = len(corrected_tb)
results = {"model": "qwen3vl-8b-base-zeroshot", "prompt_version": "qwen-specific-v2",
          "n": n, "exact_correct": exact_correct, "loose_correct": loose_correct,
          "all_exact": all_exact, "all_loose": all_loose, "per_sheet": per_sheet}
print(f"\n=== Qwen3-VL base zero-shot, Qwen-specific prompt (n={n}) ===")
print(f"  {'field':16s} {'exact':>10s} {'loose':>10s}")
for f in TB_FIELDS:
    print(f"  {f:16s} {exact_correct[f]}/{n}={exact_correct[f]/n:>4.0%} {loose_correct[f]}/{n}={loose_correct[f]/n:>4.0%}")
print(f"  {'ALL FIELDS':16s} {all_exact}/{n}={all_exact/n:>4.0%} {all_loose}/{n}={all_loose/n:>4.0%}")


  test_151: mismatch - {'drawing_number': {'pred': 'PIP-01-101', 'gt': 'PIP-01-101', 'exact': True, 'loose': True}, 'revision': {'pred': None, 'gt': None, 'exact': True, 'loose': True}, 'title': {'pred': 'Test P&ID - Starter', 'gt': 'PIP-01-101 - Test P&ID - Starter - 003', 'exact': False, 'loose': False}, 'site': {'pred': 'PIPSampleProject\nSample PIP Project\n111 Mcinnis Pkwy.\nSan Rafael California', 'gt': 'PIPSampleProject, Sample PIP Project, 111 McInnis Pkwy., San Rafael California', 'exact': False, 'loose': True}}
  test_194: mismatch - {'drawing_number': {'pred': '51Y613', 'gt': '51Y613', 'exact': True, 'loose': True}, 'revision': {'pred': '1', 'gt': 'A', 'exact': False, 'loose': False}, 'title': {'pred': 'INSTRUMENTATION GRIT WASHER NO. 1 AND GRIT DEWATERER NO. 1 P&ID', 'gt': 'INSTRUMENTATION GRIT WASHER NO. 1 AND GRIT DEWATERING UNIT NO. 1 P&ID', 'exact': False, 'loose': True}, 'site': {'pred': 'San Mateo MWRP, Nuisance Control and Wet Weather Flow Management Upgrade and Expa

## 6. Push results to HF, then disconnect

In [ ]:
import json as _json
from huggingface_hub import HfApi

with open("/content/qwen_titleblock_results.json", "w") as f:
    _json.dump(results, f, indent=2)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/qwen_titleblock_results.json",
    path_in_repo="benchmarks/qwen_titleblock_results_v2_qwen_prompt.json",
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print("results pushed to HF")

from google.colab import runtime
runtime.unassign()
